<a href="https://colab.research.google.com/github/zeyneptass/vector-databases-and-rag-fundamentals/blob/main/RAG_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pinecone sentence-transformers openai tqdm pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 16.8 MB/s eta 0:00:00


In [ ]:
import warnings
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

In [ ]:
from google.colab import userdata

# API anahtarlarını al
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# Pinecone istemcisi
pc = Pinecone(api_key=PINECONE_API_KEY)

# OpenAI istemcisi
openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("Her iki istemci de hazır.")

Her iki istemci de hazır.


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model boyutu: {model.get_embedding_dimension()}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model boyutu: 384


In [ ]:
index_name = "rag-demo"

# Varsa eski dizini sil
if index_name in pc.list_indexes().names():
    print(f"'{index_name}' siliniyor...")
    pc.delete_index(index_name)

# Yeni dizin oluştur
pc.create_index(
    name=index_name,
    dimension=model.get_embedding_dimension(),
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)
print(f"'{index_name}' oluşturuldu.")

'rag-demo' oluşturuldu.


In [ ]:
import time

while not pc.describe_index(index_name).status.ready:
    print("Dizin hazırlanıyor...")
    time.sleep(2)

print("Dizin hazır!")

Dizin hazır!


In [ ]:
index = pc.Index(index_name)
print(index)

Index(host='https://rag-demo-h0jgoc3.svc.aped-4627-b74a.pinecone.io')


In [ ]:
from datasets import load_dataset

# Eski (hatalı) kullanım:
# dataset = load_dataset("wikipedia", "20220301.simple", split="train[:2000]")

# Yeni (doğru) kullanım:
dataset = load_dataset("wikimedia/wikipedia", "20231101.simple", split="train[:2000]")

# Makale metinlerini al
articles = [item['text'] for item in dataset]
titles = [item['title'] for item in dataset]

print(f"Toplam {len(articles)} makale yüklendi.")
print("Örnek başlık:", titles[0])

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

20231101.simple/train-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  157MB            

20231101.simple/train-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/241787 [00:00<?, ? examples/s]

Toplam 2000 makale yüklendi.
Örnek başlık: April


In [ ]:
batch_size = 100

for i in tqdm(range(0, len(articles), batch_size)):
    # Bu partideki makaleleri al
    batch_articles = articles[i:i + batch_size]
    batch_titles = titles[i:i + batch_size]

    # Makaleleri vektörlere çevir
    embeddings = model.encode(batch_articles).tolist()

    # Pinecone formatına hazırla
    vectors_to_upsert = []
    for j, (title, text, embedding) in enumerate(zip(batch_titles, batch_articles, embeddings)):
        vector_id = f"wiki_{i + j}"
        metadata = {
            "title": title,
            "text": text[:1000]  # Metnin ilk 1000 karakteri (metadata boyutu için)
        }
        vectors_to_upsert.append((vector_id, embedding, metadata))

    # Pinecone'a yükle
    index.upsert(vectors=vectors_to_upsert)

print("Tüm makaleler yüklendi!")

  0%|          | 0/20 [00:00<?, ?it/s]

Tüm makaleler yüklendi!


In [ ]:
stats = index.describe_index_stats()
print(f"Toplam vektör sayısı: {stats['total_vector_count']}")

Toplam vektör sayısı: 2000


In [ ]:
def retrieve_documents(query_text, top_k=3):
    """
    Sorguya en benzer belgeleri Pinecone'dan getirir.
    """
    # Sorguyu vektöre çevir
    query_embedding = model.encode(query_text).tolist()

    # Pinecone'da ara
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )

    # Sonuçları çıkar
    documents = []
    for match in results['matches']:
        documents.append({
            "title": match['metadata']['title'],
            "text": match['metadata']['text'],
            "score": match['score']
        })

    return documents

In [ ]:
docs = retrieve_documents("What was the Berlin Wall?", top_k=3)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Belge {i} (Benzerlik: {doc['score']:.4f}) ---")
    print(f"Başlık: {doc['title']}")
    print(f"Metin: {doc['text'][:300]}...")


--- Belge 1 (Benzerlik: 0.6460) ---
Başlık: Berlin
Metin: Berlin (; ) is the capital city of Germany. It is the largest city in the European Union by population, with around 3.7 million people in 2020. Berlin residents come from about 190 different countries.

The city is in the eastern part of Germany in Central Europe and is surrounded by many forests an...

--- Belge 2 (Benzerlik: 0.5268) ---
Başlık: List of walls
Metin: This is a list of famous walls.

Lists...

--- Belge 3 (Benzerlik: 0.5266) ---
Başlık: Wall
Metin: A wall is a vertical dividing surface. It divides space in buildings into rooms or protects buildings. It is usually made of stone or brick. Walls have two main purposes: to support the top part of buildings, and to divide space, giving protection from invasion, escape, and weather.

Before powerful...


In [ ]:
def build_rag_prompt(query, documents):
    """
    Pinecone'dan gelen belgeleri kullanarak OpenAI için bir prompt oluşturur.
    """
    # Belgeleri birleştir
    context = "\n\n".join([f"Title: {doc['title']}\n{doc['text']}" for doc in documents])

    # Prompt şablonu
    prompt = f"""Answer the question based on the context below.

Context:
{context}

Question: {query}

Answer:"""

    return prompt

In [ ]:
query = "What was the Berlin Wall?"
docs = retrieve_documents(query, top_k=3)
prompt = build_rag_prompt(query, docs)

print(prompt[:1500])  # İlk 1500 karakteri göster

Answer the question based on the context below.

Context:
Title: Berlin
Berlin (; ) is the capital city of Germany. It is the largest city in the European Union by population, with around 3.7 million people in 2020. Berlin residents come from about 190 different countries.

The city is in the eastern part of Germany in Central Europe and is surrounded by many forests and lakes. Berlin has an area of . The rivers Havel, Dahme and Spree run through Berlin. It has a temperate climate.

Berlin is home to many famous buildings and monuments, like the Siegessäule, the Brandenburger Tor, the Reichstag and the boulevard Unter den Linden. On the boulevard is the Humboldt University. The city has many nightlife options.

Berlin is an important city for the history of Germany. The King of Prussia and the Emperor of Germany lived in Berlin. The government of Germany was in Berlin for many years. Bombs destroyed many buildings in the city in World War Two. The city was split into West Berlin and Ea

In [ ]:
def generate_answer(prompt):
    """
    OpenAI'a promptu gönderir ve yanıtı döndürür.
    """
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",  # Ucuz ve hızlı model
        messages=[
            {"role": "system", "content": "You are a helpful assistant that answers questions based on provided context."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=500,
        temperature=0.7
    )

    return response.choices[0].message.content

In [ ]:
# 1. Soru sor
query = "What was the Berlin Wall?"

# 2. İlgili belgeleri getir
docs = retrieve_documents(query, top_k=3)

# 3. Prompt oluştur
prompt = build_rag_prompt(query, docs)

# 4. OpenAI'dan yanıt al
answer = generate_answer(prompt)

# 5. Yanıtı yazdır
print("=" * 60)
print(f"SORU: {query}")
print("=" * 60)
print(f"\nYANIT:\n{answer}")
print("=" * 60)

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [ ]:
# Test 1: Bilim
query = "What is photosynthesis?"
docs = retrieve_documents(query, top_k=3)
prompt = build_rag_prompt(query, docs)
answer = generate_answer(prompt)
print(f"SORU: {query}\n")
print(f"YANIT:\n{answer}\n")
print("=" * 60)

# Test 2: Tarih
query = "Who was Albert Einstein?"
docs = retrieve_documents(query, top_k=3)
prompt = build_rag_prompt(query, docs)
answer = generate_answer(prompt)
print(f"SORU: {query}\n")
print(f"YANIT:\n{answer}\n")
print("=" * 60)

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}